# QSVM: an SVM with a quantum kernel
**Quantum Machine Learning · Week 2 (Quantum kernels) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qml/w2d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

The **quantum SVM** from start to finish, the honest way: split, scale, encode, build the matrices,
tune the hyperparameters by cross-validation on the training set, and open the test set once.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC

# handwritten 3s versus 8s, squeezed to 4 features with PCA: one qubit per feature
digits = load_digits()
mask = np.isin(digits.target, [3, 8])
X, y = digits.data[mask], (digits.target[mask] == 8).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pca = PCA(n_components=4).fit(X_train)                    # fitted on the training set only
scaler = MinMaxScaler((0, 1)).fit(pca.transform(X_train))
U_train = scaler.transform(pca.transform(X_train))
U_test = scaler.transform(pca.transform(X_test))
print("training points:", len(U_train), "  test points:", len(U_test), "  qubits:", U_train.shape[1])

def angle_map(x):
    qc = QuantumCircuit(len(x))
    for q, value in enumerate(x):
        qc.ry(value, q)
    return qc

def zz_map(x, reps=2):
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for q in range(n):
            qc.p(2 * x[q], q)
        for q in range(n - 1):                             # neighbors only ("linear" entanglement)
            qc.cx(q, q + 1)
            qc.p(2 * (np.pi - x[q]) * (np.pi - x[q + 1]), q + 1)
            qc.cx(q, q + 1)
    return qc

def kernels(fmap, s):
    """Training and test kernel matrices for the bandwidth s (features are multiplied by s first)."""
    S_train = np.array([Statevector(fmap(s * u)).data for u in U_train])
    S_test = np.array([Statevector(fmap(s * u)).data for u in U_test])
    return np.abs(S_train.conj() @ S_train.T) ** 2, np.abs(S_test.conj() @ S_train.T) ** 2

## Tune by cross-validation, with precomputed kernels
For each fold, train on the training rows and columns and validate on the validation rows against the
training columns: the same "train versus train, others versus train" rule as yesterday.

In [ ]:
def cv_accuracy(K, y, C, folds=5):
    scores = []
    for tr, va in StratifiedKFold(folds, shuffle=True, random_state=0).split(K, y):
        model = SVC(kernel="precomputed", C=C).fit(K[np.ix_(tr, tr)], y[tr])
        scores.append(model.score(K[np.ix_(va, tr)], y[va]))
    return np.mean(scores)

results = {}
for name, fmap in [("angle", angle_map), ("ZZ", zz_map)]:
    best = None
    for s in [0.25, 0.5, 1.0, 2.0, np.pi]:
        K_train, K_test = kernels(fmap, s)
        for C in [0.1, 1, 10, 100]:
            score = cv_accuracy(K_train, y_train, C)
            if best is None or score > best[0]:
                best = (score, s, C, K_train, K_test)
    score, s, C, K_train, K_test = best
    model = SVC(kernel="precomputed", C=C).fit(K_train, y_train)
    results[name] = (model, K_train, K_test)
    print(f"{name:5s}: best CV accuracy {score:.3f} with s = {s:.2f}, C = {C}  ->  test accuracy {model.score(K_test, y_test):.3f}")

## What did the SVM learn?
Its decision is a weighted vote of the **support vectors**: $f(x) = \sum_i \alpha_i y_i\, k(x_i, x) + b$. Rebuild it
by hand from the kernel values and compare with scikit-learn.

In [ ]:
model, K_train, K_test = results["ZZ"]
print(f"support vectors: {len(model.support_)} of {len(U_train)} training points")
by_hand = K_test[:, model.support_] @ model.dual_coef_.ravel() + model.intercept_[0]
print("decision function rebuilt by hand matches:", np.allclose(by_hand, model.decision_function(K_test)))

## Your turn: tune the classical baseline just as carefully
A fair comparison gives the classical model the same care. Tune an RBF SVM on the same four PCA features
with `GridSearchCV` over `C` and `gamma` (5-fold), and compare its test accuracy with the quantum kernels'.

In [ ]:
grid = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.1, 1, 10]}
search = None        # TODO: GridSearchCV(SVC(kernel="rbf"), grid, cv=5).fit(U_train, y_train)
# TODO: print the best parameters, the best CV score and the test score

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
grid = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.1, 1, 10]}
search = GridSearchCV(SVC(kernel="rbf"), grid, cv=5).fit(U_train, y_train)
print(f"RBF: best {search.best_params_}, CV accuracy {search.best_score_:.3f}, test accuracy {search.score(U_test, y_test):.3f}")
for name, (model, K_train, K_test) in results.items():
    print(f"{name:5s} kernel: test accuracy {model.score(K_test, y_test):.3f}")
print("All within two percent: two test images out of 108, far too small a gap to call a winner. On this")
print("task the quantum kernels are as good as a well-tuned classical one, not clearly better: a typical,")
print("honest result.")